In [1]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `~/.julia/dev/drag-formation`


In [2]:
using LinearAlgebra
using SparseArrays
using Convex
using GLPK, MathOptInterface, JuMP

In [3]:
### Constants
const μ_mks = 398600.44 #Earth standard gravitational parameter (km^3/sec^2)
const J2 = 0.00108262668 #Earth J2 gravity coefficient (dimensionless)const 
const Re_mks = 6378.137 #Radius of the Earth (km)
const Ωe_mks = 7.292115e-5 #Rotation rate of Earth (rad/sec)

const Cd = 2.2 #Drag coefficient (unitless)
const AoM_mks = (2+sqrt(2))*3e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
const AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg

In [4]:
#Orbital Parameters (roughly ISS)
const a_mks = 6791.0 #semi-major axis (km)
const inc = 54.74 #inclination (deg)
const n_mks = sqrt(μ_mks/a_mks^3) #mean motion (rad/sec)
const P_mks = 2*pi/n_mks; #orbital period (sec) 1.54 hour 5569.440609858254 sec


In [5]:
#Normalize units
const a = 1.0 #distance unit is semi-major axis
const P = 1 #time unit is orbital period
const μ = (2*pi/P)^2 * a^3 #μ_mks*(P_mks^2/a_mks^3)
const n = sqrt(μ/a^3)
const Re = Re_mks/a_mks
const Ωe = Ωe_mks*P_mks;

const AoM = AoM_mks/(a_mks^2)
const AoM0 = AoM0_mks/(a_mks^2);


In [6]:
### Coordinate and unit transformations

function cartesian_to_spherical(x)
    r = sqrt(x[1:3]'*x[1:3]) #range
    ϕ = atan(x[2],x[1]) #azimuth
    θ = atan(x[3],sqrt(x[1:2]'*x[1:2])) #elevation measured up from equatorial plane
    
    return [r; ϕ; θ]
end

function spherical_to_cartesian(s)
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    x = r*cos(θ)*cos(ϕ)
    y = r*cos(θ)*sin(ϕ)
    z = r*sin(θ)
    
    return [x; y; z]
end

function rtn_frame(x0)
    r = x0[1:3]
    v = x0[4:6]
    n = cross(r,v)
    
    R = r/norm(r)
    N = n/norm(n)
    T = cross(N,R)
    
    return [R T N] #rotation matrix from RTN to Cartesian frame
end

function rtn_to_cartesian(x0,δr)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    r = R*δr + r0 #position in inertial frame
    
    return r
end

function cartesian_to_rtn(x0,r)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    δr = R'*(r - r0) #position in RTN frame
    
    return δr
end

function cartesian_to_aθΩ(x)
    r = x[1:3]
    v = x[4:6]
        
    E = 0.5*v'*v + potential(cartesian_to_spherical(x))
    
    a = -μ/(2*E)
    
    i_x = [1,0,0]
    i_y = [0,1,0]
    i_z = [0,0,1]
    
    h = cross(r,v)
    Ω = atan(h[2]/norm(h[1:2]), h[1]/norm(h[1:2]))
    
    n = cross(i_z,h)/norm(cross(i_z,h))
    if dot(r,i_z) > 0
        θ = acos(dot(n,(r/norm(r))))
    else
        θ = 2*pi- acos(dot(n,(r/norm(r))))
    end
    return [a; θ; Ω]
end

function cartesian_to_ΔaθΩ(x0,x1)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    r1 = x1[1:3]
    v1 = x1[4:6]
    
    E0 = 0.5*v0'*v0 + potential(cartesian_to_spherical(x0))
    E1 = 0.5*v1'*v1 + potential(cartesian_to_spherical(x1))
    
    a0 = -μ/(2*E0)
    a1 = -μ/(2*E1)
    Δa = a1-a0
    

    R = rtn_frame(x0) #rotation from RTN to Cartesian frame    
    rtn1 = R'*r1
    Δθ = atan(rtn1[2]/norm(rtn1[1:2]), rtn1[1]/norm(rtn1[1:2]))
    
    h0 = cross(r0,v0)
    Ω0 = atan(h0[2]/norm(h0[1:2]), h0[1]/norm(h0[1:2]))
    
    h1 = cross(r1,v1)
    Ω1 = atan(h1[2]/norm(h1[1:2]), h1[1]/norm(h0[1:2]))
    
    ΔΩ = Ω1-Ω0
    
    return [Δa; Δθ; ΔΩ]
end

function mks_to_orbital(x)
    #change state from MKS units to orbital units
    
    return [x[1:3]/a_mks; x[4:6]*(P_mks/a_mks)]
end

function orbital_to_mks(x)
    #change state from orbital units to MKS
    
    return [x[1:3]*a_mks; x[4:6]*(a_mks/P_mks)]
end

orbital_to_mks (generic function with 1 method)

In [7]:
function density_mks(alt)
    #This fit is good from 250-500 km altitude
    #Units are km, kg
    a = 106.2
    b = -0.03243
    c = 1.988
    d = -0.01672
    ρ = a*exp(b*alt) + c*exp(d*alt)
end

function atm_density(r)
    #ρ = ρ0*exp((a0-r)/H)
    ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
end

atm_density (generic function with 1 method)

In [8]:
#Continuous-Time Linear model
#state is [Δθ; Δa; ΔΩ] where these are differences between satellite 1 and satellite 2
#Can treat drag input as positive part going to sat 2 and negative part going to sat 1
# delta is always with respect to sat 1
function kvalues(a)
    n = sqrt(μ/a^3)
    Fd = -0.5*Cd*atm_density(a)*(AoM)*(n*a)^2*0.965#-15/a_mks
    ȧ = 2*(a^2/μ)*(a)*n*Fd
    k1 = (21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*cosd(inc) #dΩ̇/da
    k2 = -(3.0/2)*sqrt(μ/a^5) #dθ̇/da (dn/da) #(21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*(1-1.5*(sind(inc))^2)#
    k3 = ȧ #dȧ/d(Drag) - 3*sqrt(μ*a)*J2*Re*Re/a^3*sind(inc)^2 
    k4 = k1/k2
    return [k1, k2, k3, k4]
end

kvalues (generic function with 1 method)

In [9]:
#Discrete-time model
num_sat = 1 # number of couple of satellites 


function create_continuous(a, num_sat)
    #Discrete-time model
#     num_sat = 4 # number of couple of satellites 
    k1, k2, k3, k4 = kvalues(a)
    A0 = [0 k2 0; 0 0 0; 0 k1 0]
    A_bar = zeros(3*num_sat,3*num_sat) 
    B_bar = zeros(3*num_sat,num_sat+1) 

    # B_bar[1:3,1:2]=B0
    for k =0:num_sat-1
        A_bar[1+(3*k):3+(3*k),1+(3*k):3+(3*k)] = A0
        B_bar[3*k+2,1] = -k3
        B_bar[3*k+2,2+k] = k3
    end

    return A_bar, B_bar 
end

function c2d(a,dt,num_sat)
    # discretize a continuous time linear system 
    # convert xdot = Ax + Bu 
    # into x_{k+1} = Ad x_k + Bd x_k 
    # where we have dt between k and k +1 
    
    A,B = create_continuous(a, num_sat)
    nx,nu = size(B)
    
    H = exp(dt*[A B;zeros(nu, nx + nu)])
    Ad = H[1:nx, 1:nx]
    Bd = H[1:nx, (nx + 1):(nx + nu)]
    return Ad, Bd 
end


const h =0.5 #time step
Ad, Bd = c2d(a, h, num_sat)
# A = I + h*A_bar
# B = h*B_bar + 0.5*h*h*A_bar*B_bar;
# B_bar

In [10]:
function sat_dynamics_rk4(x,u,h)
    f1 = satellite_dynamics(x,u)
    f2 = satellite_dynamics(x + 0.5*h*f1,u)
    f3 = satellite_dynamics(x + 0.5*h*f2, u)
    f4 = satellite_dynamics(x + h*f3, u)
    return x + (h/6.0)*(f1 + 2*f2 +2*f3 +f4)
end

sat_dynamics_rk4 (generic function with 1 method)

In [11]:
### Full Orbital Dynamics with J2 and Drag
function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end


function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r1 = x[1:3]
    v1 = x[4:6]
    r2 = x[7:9]
    v2 = x[10:12]
    u1 = u[1][1]
    u2 = u[2][1]

    
    pos1_x = r1[1]
    pos1_y = r1[2]
    pos1_z = r1[3]
    r1_norm = sqrt(r1[1]^2+r1[2]^2+r1[3]^2)
    
    pos2_x = r2[1]
    pos2_y = r2[2]
    pos2_z = r2[3]
    r2_norm = sqrt(r2[1]^2+r2[2]^2+r2[3]^2)
    
    g1 = +1.5*J2*μ_mks*Re_mks^2/r1_norm^4*[
        pos1_x/r1_norm*(5*pos1_z^2/r1_norm^2-1);
        pos1_y/r1_norm*(5*pos1_z^2/r1_norm^2-1);
        pos1_z/r1_norm*(5*pos1_z^2/r1_norm^2-3)] - μ_mks*r1/(r1_norm^3)
    
    g2 = +1.5*J2*μ_mks*Re_mks^2/r2_norm^4*[
        pos2_x/r2_norm*(5*pos2_z^2/r2_norm^2-1);
        pos2_y/r2_norm*(5*pos2_z^2/r2_norm^2-1);
        pos2_z/r2_norm*(5*pos2_z^2/r2_norm^2-3)] - μ_mks*r2/(r2_norm^3)
    
    
    v1_rel = v1 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
#         ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
    ρ1 = density_mks(r1_norm - Re_mks)
    Fdrag1 = -0.5*(u1*AoM_mks)*Cd*ρ1*(norm(v1_rel))*v1_rel
    
    v2_rel = v2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ2 = density_mks(r2_norm - Re_mks)
    Fdrag2 = -0.5*(u2*AoM_mks)*Cd*ρ2*(norm(v2_rel))*v2_rel
#     println("u1 ", u1, " r1_norm - Re_mks ",r1_norm - Re_mks," Fdrag1 ", norm(Fdrag1), " ρ1 ", ρ1, " v1_rel ", v1_rel)
#     println(" r2_norm - Re_mks ",r2_norm - Re_mks," Fdrag2 ", norm(Fdrag2), " ρ2 ", ρ2, " v2_rel ", v2_rel)
    
    v̇1 = g1 + Fdrag1
    v̇2 = g2 + Fdrag2
    return [v1; v̇1; v2; v̇2]
end


satellite_dynamics (generic function with 1 method)

In [12]:
#Initial and terminal constraints and #Inequality constraints
params = (
    l_12 = 1.0, # number of revolutions
    l_13 = 6.0,
    l_14 = 8.0,
    l_15 = 4.0,
    l_16 = 5.0,
    Δx0 = zeros(size(Ad,1)),
    Δaf = 0, #end at the same altitude
    u_min = 0,
    u_max = 1,
    Δa_min = -10/a_mks /num_sat,
    Δa_max = 10/a_mks /num_sat,) 

k4 = kvalues(a)[4]
params = (; params..., Δθf = 2*pi*[params.l_12,params.l_13,params.l_14,params.l_15,params.l_16],
)

params = (; params..., ΔΩf = k4*(params.Δθf))# Right Ascension Separation


(l_12 = 1.0, l_13 = 6.0, l_14 = 8.0, l_15 = 4.0, l_16 = 5.0, Δx0 = [0.0, 0.0, 0.0], Δaf = 0, u_min = 0, u_max = 1, Δa_min = -0.0014725371815638344, Δa_max = 0.0014725371815638344, Δθf = [6.283185307179586, 37.69911184307752, 50.26548245743669, 25.132741228718345, 31.41592653589793], ΔΩf = [-0.012123811564198254, -0.07274286938518952, -0.09699049251358603, -0.048495246256793016, -0.06061905782099127])

In [13]:
#Discretized Optimal Control Problem Parameters
const Nx = size(Ad,1)     # number of state
const Nu = size(Bd,2)     # number of controls
const Tfinal = 800#Int(round(800*P_mks)) # final time (number of orbits)
const Nt = Int(Tfinal/h)+1    # number of time steps
const Nz = Nt*(Nx+Nu) #number of decision variables
thist = Array(range(0,h*(Nt-1), step=h));

#Useful Matrices

# decision variables are stacked z = [u1; x2; u2; x3; u3; ...; xf]
U1 = [sparse(kron(Diagonal(I,Nt), sparse([I zeros(1,(Nu-1+Nx))]))) for i = 1:num_sat] #Matrix that picks out all u of sat 1
U2 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,i) I zeros(1,(Nu-i)+Nx-1)]))) for i = 1:num_sat] #Matrix that picks out all u of sat n
X1 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-1-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x1 = Δθ
X2 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,1) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-2-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x2 = Δa
X3 = [sparse(kron(Diagonal(I,Nt), sparse([zeros(1,Nu) zeros(1,2) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-3-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x3 = ΔΘ


X1F = spzeros(num_sat,(Nu+Nx)*Nt);  #Matrix that picks out x1f = Δθf
X2F = spzeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x2f = Δaf
# X3F = zeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x3f = ΔΘf

for i = num_sat:-1:1
    X1F[num_sat-i+1,end-(i)*3+1] = 1.0
    X2F[num_sat-i+1,end-(i)*3+2] = 1.0
#     X3F[num_sat-i+1,end-(i)*3+3] = 1.0
end

# z = Variable(Nz)

# #Convex Formulation
# objective = 0
# Ac = U1[1] 
# b = zeros(Nt + Nt*num_sat + Nt*num_sat)
# for i in 1:num_sat
#     Ac  = [Ac; U2[i];]
# end

# for i in 1:num_sat
#     Ac  = [Ac; X1[i]]
#     b[Nt + Nt*num_sat + (i-1)*Nt+ 1:Nt + Nt*num_sat + (i)*Nt] .= params.Δθf[i]
# end
# objective += norm(Ac*z-b,1)# + sum(norm(X1[:,:,i]*z - Δθf[i],1) for i in 1:num_sat);

# problem = minimize(objective);
# Int(round(800*P_mks)

In [14]:
# using MosekTools
function LP_controller(a,Δx, params)
    z = Variable(Nz)
    # Optimizer
    optimizer = GLPK.Optimizer()
    MOI.set(optimizer, MOI.RawParameter("msg_lev"), GLPK.GLP_MSG_ON)
    
    #Dynamics equality constraints C*z = d
    Ad, Bd = c2d(a, h, num_sat)
    C = sparse([[Bd -I zeros(Nx,(Nt-1)*(Nu+Nx))]; zeros(Nx*(Nt-1),Nu) [kron(Diagonal(I,Nt-1), [Ad Bd]) zeros((Nt-1)*Nx,Nx)] + [zeros((Nt-1)*Nx,Nx) kron(Diagonal(I,Nt-1),[zeros(Nx,Nu) Diagonal(-I,Nx)])]])
    d = [-Ad*Δx; zeros(Nx*(Nt-1))];

    #Convex Formulation
    objective = 0
    Ac = U1[1] 
    b = zeros(Nt + Nt*num_sat + Nt*num_sat)
    for i in 1:num_sat
        Ac  = [Ac; U2[i];]
    end

    for i in 1:num_sat
        Ac  = [Ac; X1[i]]
        b[Nt + Nt*num_sat + (i-1)*Nt+ 1:Nt + Nt*num_sat + (i)*Nt] .= params.Δθf[i]
    end
    objective += norm(Ac*z-b,1)# + sum(norm(X1[:,:,i]*z - Δθf[i],1) for i in 1:num_sat);

    problem = minimize(objective);
    problem.constraints += [C*z == d] #dynamics constraints

    for i = 1:num_sat
        problem.constraints += [X2F[i,:]'*z == params.Δaf] #final altitude constraint
    #     problem.constraints += [X3F[i,:]'*z == ΔΩf] #final altitude constraint
        problem.constraints += [U1[i]*z >= params.u_min; U1[i]*z <= params.u_max] #drag constraints
        problem.constraints += [U2[i]*z >= params.u_min; U2[i]*z <= params.u_max] #drag constraints
        problem.constraints += [X2[i]*z >= params.Δa_min; X2[i]*z <= params.Δa_max]; #altitude constraints
    end
#     solve!(problem, Mosek.Optimizer)
    solve!(problem,optimizer; silent_solver = false)
    
    u1 = zeros(size(U1[1])[1],num_sat) # sat 1 control
    u2 = zeros(size(U2[1])[1],num_sat) # sat n control
    for i = 1:num_sat
        u1[:,i] = U1[i]*z.value;
        u2[:,i] = U2[i]*z.value;
    end
    return [u1[1,:],u2[1,:]]
end

LP_controller (generic function with 1 method)

In [15]:
r0 = [a_mks; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = [r0; v0; r0; v0];

# initialization
u1hist = zeros(size(U1[1])[1],num_sat)
u2hist = zeros(size(U2[1])[1],num_sat)
θhist = zeros(size(X1[1])[1]+1,num_sat)
ahist = zeros(size(X2[1])[1]+1,num_sat)
Ωhist = zeros(size(X3[1])[1]+1,num_sat)
x_hist = zeros(length(x0),Nt)
x_after = zeros(length(x0))
x_hist[:,1] = x0
xk = x0


x0c = cartesian_to_ΔaθΩ(mks_to_orbital(x0[1:6]),mks_to_orbital(x0[7:12]))
xc = x0c
#find first controller
# u0 = LP_controller(a, x0c, params)
#store first step
ahist[1,:] .= x0c[1]
θhist[1,:] .= x0c[2]
Ωhist[1,:] .= x0c[3]

# print(u0[:])
for k = 1:(Nt-1)
    println(k)
    uk = LP_controller(a, xc, params)
    x_prev = x_hist[:,k]
    for kk = 1:P_mks*h # solve the dynamics for a full period
        x_after = sat_dynamics_rk4(x_prev,uk,h)
        x_prev = x_after
#         println(kk)
    end
#     println(x_after)
    xc = cartesian_to_ΔaθΩ(mks_to_orbital(x_after[1:6]),mks_to_orbital(x_after[7:12]))
    x_hist[:,k+1] = x_after
    u1hist[k,1] = uk[1][1]
    u2hist[k,1] = uk[2][1]
    ahist[k,1] = xc[1]
    θhist[k,1] = xc[2]
    Ωhist[k,1] = xc[3]
#     xk = sat_dynamics_rk4(xk,uk,h)  #simulate dynamics
#     println("xk ",x_hist[:,k+1], " uk ", uk)
    temp = abs(norm(x_hist[1:3,k+1])-norm(x_hist[7:9,k+1]))
    println("uk ", uk, " Δa ",xc[1], " a ",a )
end

1
      0: obj =   0.000000000e+00 inf =   1.006e+04 (1601)
   9483: obj =   1.236830131e+03 inf =   4.286e+03 (1171) 31
  11268: obj =   5.604266453e+03 inf =   1.550e-02 (0) 174
  15301: obj =   5.109200093e+03 inf =   5.037e-11 (0) 149
* 15631: obj =   5.108123168e+03 inf =   5.037e-11 (0) 137
uk [[0.0], [1.0]] Δa -1.8222407816015362e-6 a 1.0
2
      0: obj =   0.000000000e+00 inf =   1.006e+04 (1603)
  10097: obj =   0.000000000e+00 inf =   6.283e+00 (2) 51
  11968: obj =   1.005105601e+04 inf =   1.533e-05 (0) 19
  17423: obj =   5.108747344e+03 inf =   5.591e-11 (0) 169
* 17753: obj =   5.107735741e+03 inf =   5.037e-11 (0) 137
uk [[0.0], [1.0]] Δa -3.887381986533711e-6 a 1.0
3
      0: obj =   0.000000000e+00 inf =   1.006e+04 (1603)
   9994: obj =   0.000000000e+00 inf =   6.283e+00 (2) 48
  12597: obj =   1.006365782e+04 inf =   7.492e-06 (0) 23
  17657: obj =   5.116542362e+03 inf =   1.217e-10 (0) 166
* 17661: obj =   5.116542362e+03 inf =   1.209e-10 (0)
uk [[0.0], [1.0]] Δ

┌ Warning: Problem status INFEASIBLE; solution may be inaccurate.
└ @ Convex /home/josephine/.julia/packages/Convex/uI27T/src/solution.jl:263


LoadError: MethodError: no method matching *(::SparseMatrixCSC{Float64, Int64}, ::Nothing)
[0mClosest candidates are:
[0m  *(::Any, ::Any, [91m::Any[39m, [91m::Any...[39m) at ~/julia-1.7.3/share/julia/base/operators.jl:655
[0m  *(::Union{SparseArrays.AbstractSparseMatrixCSC{Tv, Ti}, SubArray{Tv, 2, <:SparseArrays.AbstractSparseMatrixCSC{Tv, Ti}, Tuple{Base.Slice{Base.OneTo{Int64}}, I}} where I<:AbstractUnitRange, LowerTriangular{Tv, <:Union{SparseArrays.AbstractSparseMatrixCSC{Tv, Ti}, SubArray{Tv, 2, <:SparseArrays.AbstractSparseMatrixCSC{Tv, Ti}, Tuple{Base.Slice{Base.OneTo{Int64}}, I}} where I<:AbstractUnitRange}}, UpperTriangular{Tv, <:Union{SparseArrays.AbstractSparseMatrixCSC{Tv, Ti}, SubArray{Tv, 2, <:SparseArrays.AbstractSparseMatrixCSC{Tv, Ti}, Tuple{Base.Slice{Base.OneTo{Int64}}, I}} where I<:AbstractUnitRange}}} where {Tv, Ti}, [91m::SubArray{Tv, 1, <:SparseArrays.AbstractSparseMatrixCSC{Tv, Ti}, Tuple{Base.Slice{Base.OneTo{Int64}}, Int64}, false} where {Tv, Ti}[39m

In [ ]:
# r0 = [a_mks; 0; 0]
# v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
# x0 = [r0; v0; r0; v0];

# # initialization
# u1hist = zeros(size(U1[1])[1],num_sat)
# u2hist = zeros(size(U2[1])[1],num_sat)
# θhist = zeros(size(X1[1])[1]+1,num_sat)
# ahist = zeros(size(X2[1])[1]+1,num_sat)
# Ωhist = zeros(size(X3[1])[1]+1,num_sat)

# #convert initial condition
# # x0_orbital = zeros(1,12)
# # x0_orbital[1:6] = mks_to_orbital(x0[1:6])
# # x0_orbital[7:12] = mks_to_orbital(x0[7:12])

# # x0c = cartesian_to_ΔaθΩ(mks_to_orbital(x0[1:6]),mks_to_orbital(x0[7:12]))

# # a1 = cartesian_to_aθΩ(x0_orbital[1:6])[1] 
# # a2 = cartesian_to_aθΩ(x0_orbital[7:12])[1] 

# # find first controller
# # u0 = LP_controller(a, x0c, params)
# # println(x0c)
# # println(u0)
# # println(zeros(size(Ad,1)))
# #store first step
# # u1hist[1,:] .= u0[1]
# # u2hist[1,:] .= u0[2]
# # θhist[1,:] .= x0c[1]
# # ahist[1,:] .= x0c[2]
# # Ωhist[1,:] .= x0c[3]

# xk = x0
# # xc = x0c
# for k = 1:(Nt-1)
#     println(k)
# #     uk = LP_controller(a,xc,params)  # find control
#     uk = [0.0, 1.0]
#     xk = sat_dynamics_rk4(xk,uk,h)  #simulate dynamics
#     println("xk ",xk, " uk ", uk)
# #     xk_orbital = zeros(1,12)
# #     xk_orbital[1:6] = mks_to_orbital(xk[1:6]) #transform dynamics output first sat into normalized coordinates
# #     xk_orbital[7:12] = mks_to_orbital(xk[7:12])  #transform dynamics output second sat into normalized coordinates
# #     xc = cartesian_to_ΔaθΩ(mks_to_orbital(xk[1:6]),mks_to_orbital(xk[7:12])) # tranform output into difference for control
# #     println("xc ",xc[1], " uk ", uk)
# #     a1 = cartesian_to_aθΩ(xk[1:6])[1] # find new a to update control dynamics
# #     a2 = cartesian_to_aθΩ(xk[7:12])[1] #  find new a to update control dynamics
# #     println("a1 ", a1, " a2 ", a2)
#     temp = abs(norm(xk[1:3])-norm(xk[7:9]))
#     println("a ",temp )
# #     u1hist[k,1] = uk[1]
# #     u2hist[k,1] = uk[2]
# #     θhist[k,1] = xc[1]
# #     ahist[k,1] = temp#xc[2]
# #     Ωhist[k,1] = xc[3]
# end

In [ ]:
# res = LP_controller(params.Δx0, params)
# u1hist = zeros(size(U1[1])[1],num_sat) # sat 1 control
# u2hist = zeros(size(U2[1])[1],num_sat) # sat n control
# θhist = zeros(size(X1[1])[1]+1,num_sat)
# ahist = zeros(size(X2[1])[1]+1,num_sat)
# Ωhist = zeros(size(X3[1])[1]+1,num_sat)
# for i = 1:num_sat
#     u1hist[:,i] = U1[i]*res;
#     u2hist[:,i] = U2[i]*res;
#     θhist[:,i] = [params.Δx0[1]; X1[i]*res];
#     ahist[:,i] = [params.Δx0[2]; X2[i]*res];
#     Ωhist[:,i] = [params.Δx0[3]; X3[i]*res];
# end
sqrt(6110.304503150477^2 + 1709.598434949998^2 + 2417.8918430078197^2)-Re_mks


In [ ]:
using Plots
plotlyjs()

mylabel = String[]
mylabel1 = String[]
mylabel2 = String[]
for i = 1:num_sat
    append!(mylabel,["Sat. 1-"*string(i+1)])
    append!(mylabel1,["Sat. 1"])
    append!(mylabel2,["Sat. "*string(i+1)])
end

plot(thist,u1hist,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(thist,u2hist,label=permutedims(mylabel2))
plot!(xlabel="Time, s", ylabel = "Control")

In [ ]:
plot(thist[1:end],broadcast.(rad2deg,θhist[2:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "True Anomaly Difference, deg")

In [ ]:
plot(thist,a_mks*ahist[2:end,:], label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Semi-major axis Difference, km")


In [ ]:
plot(thist,broadcast.(rad2deg,Ωhist[2:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "RAAN Difference, deg")

In [ ]:
plot(thist,a_mks*Ωhist[2:end,:],label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [ ]:
### Full Orbital Dynamics with J2 and Drag

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    r_noJ2 = x[7:9]
    v_noJ2 = x[10:12]

    pos_x = r[1]
    pos_y = r[2]
    pos_z = r[3]
    r_norm = sqrt(r[1]^2+r[2]^2+r[3]^2)
    r_norm_noJ2 = sqrt(r_noJ2[1]^2+r_noJ2[2]^2+r_noJ2[3]^2)
    
    g = +1.5*J2*μ_mks*Re_mks^2/r_norm^4*[
        pos_x/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_y/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_z/r_norm*(5*pos_z^2/r_norm^2-3)] - μ_mks*r/r_norm^3
    
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = density_mks(r_norm - Re_mks)
    Fdrag = -0.5*(u*AoM_mks)*Cd*ρ*(sqrt(v_rel[1]^2+v_rel[2]^2+v_rel[3]^2))*v_rel

    v_rel_noJ2 = v_noJ2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ_noJ2 = density_mks(r_norm_noJ2 - Re_mks)
    Fdrag_noJ2 = -0.5*(u*AoM_mks)*Cd*ρ_noJ2*(sqrt(v_rel_noJ2[1]^2+v_rel_noJ2[2]^2+v_rel_noJ2[3]^2))*v_rel_noJ2

    v̇ = g + Fdrag
    v̇_noJ2 = -μ_mks/(r_norm_noJ2)^3 * r_noJ2 +  Fdrag_noJ2
    return [v; v̇; v_noJ2; v̇_noJ2]
end


In [ ]:
#Initial conditions in Cartesian Coordinates
# r0 = [a; 0; 0]
# v0 = [0; cosd(inc)*sqrt(μ/a); sind(inc)*sqrt(μ/a)]
# x0 = [r0; v0; r0; v0];

# r1 = [a; 0; 0]
# v1 = [0; cosd(inc)*sqrt(μ/(a)); sind(inc)*sqrt(μ/(a))]
# x1 = [r1; v1; r1; v1];

# tspan = (0.0, Tfinal+50)
# Nt_t = Tfinal+150

r0 = [a_mks; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = [r0; v0; r0; v0];

r1 = [a_mks; 0; 0]
v1 = [0; cosd(inc)*sqrt(μ_mks/(a_mks)); sind(inc)*sqrt(μ_mks/(a_mks))]
x1 = [r1; v1; r1; v1];

step_t = 60*10 # save results every 10 mins
tspan = (0.0, Int(round((Tfinal)*P_mks/step_t)*step_t))
Nt_t = Int(round((Tfinal)*P_mks/step_t))-1#/100

In [ ]:
using Interpolations
#Full Orbit Mechanics Simulation and save results
# Nt = 150
# Nt = Nt*3
temp = LinRange(tspan[1],tspan[2],Nt_t)
x0_soln = zeros(6*num_sat,Nt_t)
x1_soln = zeros(6*num_sat,Nt_t)

x0_noJ2_soln = zeros(6*num_sat,Nt_t)
x1_noJ2_soln = zeros(6*num_sat,Nt_t)

Δa_soln = zeros(num_sat,Nt_t)
Δθ_soln = zeros(num_sat,Nt_t)
ΔΩ_soln = zeros(num_sat,Nt_t)

r_soln1 = zeros(num_sat,Nt_t)
ϕ_soln1 = zeros(num_sat,Nt_t)
θ_soln1 = zeros(num_sat,Nt_t)
u_soln1 = zeros(num_sat,Nt_t)

r_soln2 = zeros(num_sat,Nt_t)
ϕ_soln2 = zeros(num_sat,Nt_t)
θ_soln2 = zeros(num_sat,Nt_t)
u_soln2 = zeros(num_sat,Nt_t)

a_soln1 = zeros(num_sat,Nt_t)
ν_soln1 = zeros(num_sat,Nt_t)
Ω_soln1 = zeros(num_sat,Nt_t)

a_soln2 = zeros(num_sat,Nt_t)
ν_soln2 = zeros(num_sat,Nt_t)
Ω_soln2 = zeros(num_sat,Nt_t)

r_soln1_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln1_noJ2 = zeros(num_sat,Nt_t)
θ_soln1_noJ2 = zeros(num_sat,Nt_t)
u_soln1_noJ2 = zeros(num_sat,Nt_t)

r_soln2_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln2_noJ2 = zeros(num_sat,Nt_t)
θ_soln2_noJ2 = zeros(num_sat,Nt_t)
u_soln2_noJ2 = zeros(num_sat,Nt_t)

h_traj = 1#step_t#/P_mks
thist_c = thist.*P_mks
for i = 1:num_sat
    u1interp = 0;#LinearInterpolation(thist_c, u1hist[:,i])
    u2interp = 1;#LinearInterpolation(thist_c, u2hist[:,i])

    x_hist1 = zeros(length(x0),Int(tspan[2]/h_traj))
    x_hist2 = zeros(length(x1),Int(tspan[2]/h_traj))
    x_hist1[:,1] = x0 
    x_hist2[:,1] = x1 
    temp = LinRange(tspan[1],tspan[2],Nt_t)
    k = 0 
    save_index = 0
    for t = tspan[1]:h_traj:tspan[2]-step_t# temp# 1:100:tspan[2]-1
        k += 1
        #Stored Controller
        if t < tspan[2]
            u1 = 0;#u1interp(t) 
            u2 = 1;#u2interp(t) 
        else
            u1 = 0;#u_soln1[i,k]
            u2 = 0;#u_soln2[i,k]
        end

        x_hist1[:,k+1] .= sat_dynamics_rk4(x_hist1[:,k],u1,h_traj)
        x_hist2[:,k+1] .= sat_dynamics_rk4(x_hist2[:,k],u2,h_traj)

        # save results
        if k%step_t == 0
            save_index += 1
            u_soln1[i,save_index] = u1
            u_soln2[i,save_index] = u2
            x0_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[1:6,k]
            x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[7:12,k]
        
            x1_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[1:6,k]
            x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[7:12,k]
    
            rϕθ1_noJ2 = cartesian_to_spherical(x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2_noJ2 = cartesian_to_spherical(x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ1 = cartesian_to_spherical(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2 = cartesian_to_spherical(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            ΔaθΩ = cartesian_to_ΔaθΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]), mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ1 = cartesian_to_aθΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ2 = cartesian_to_aθΩ(mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
    
    
            Δa_soln[i,save_index] = ΔaθΩ[1]
            Δθ_soln[i,save_index] = ΔaθΩ[2]
            ΔΩ_soln[i,save_index] = ΔaθΩ[3]

            a_soln1[i,save_index] = aθΩ1[1]
            ν_soln1[i,save_index] = aθΩ1[2]
            Ω_soln1[i,save_index] = aθΩ1[3]
        
            a_soln2[i,save_index] = aθΩ2[1]
            ν_soln2[i,save_index] = aθΩ2[2]
            Ω_soln2[i,save_index] = aθΩ2[3]
        
            r_soln1_noJ2[i,save_index] = rϕθ1_noJ2[1]
            ϕ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[2]
            θ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[3]
    
            r_soln2_noJ2[i,save_index] = rϕθ2_noJ2[1]
            ϕ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[2]
            θ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[3]
        
            r_soln1[i,save_index] = rϕθ1[1]
            ϕ_soln1[i,save_index] = rϕθ1[2]
            θ_soln1[i,save_index] = rϕθ1[3]
    
            r_soln2[i,save_index] = rϕθ2[1]
            ϕ_soln2[i,save_index] = rϕθ2[2]
            θ_soln2[i,save_index] = rϕθ2[3]
        end

    end
end
tsamp = LinRange(tspan[1],tspan[2],Nt_t);


In [ ]:
using Plots
# tsamp = LinRange(tspan[1],tspan[2],Nt)
plot(tsamp[1:30:end]./(60*60*24),x0_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[3,1:30:end],color=:green)

plot!(tsamp[1:30:end]./(60*60*24),x1_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[3,1:30:end],color=:green)

# plot(tsamp,x0_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x0_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x0_noJ2_soln[3,:],color=:green)

# plot!(tsamp,x1_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x1_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x1_noJ2_soln[3,:],color=:green)
# x1_soln[3,:]

In [ ]:
plot(tsamp[1:30:end]./(60*60*24),broadcast.(rad2deg,ν_soln1[1,1:30:end]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln1[1,:]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln2[1,:]),color=:blue)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

# length(Ω_soln1[1,:][Ω_soln1[1,:] .>3.])

In [ ]:
tsamp = LinRange(tspan[1],tspan[2],Nt_t)
plot(tsamp[1:30:end]./(60*60*24),Δa_soln[1:30:end]',label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "Altitude Difference, km")


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Δθ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "True Anomaly Difference, deg")

In [ ]:
plot(tsamp./(60*40*24),broadcast.(rad2deg,ΔΩ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "RAAN Difference, deg", ylim = [-10,5])

In [ ]:
plot(tsamp,a_mks*ΔΩ_soln',label=permutedims(mylabel), ylim = [-100*1e1,10*1e2])
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [ ]:
plot(tsamp./(60*60*24),u_soln1',color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),u_soln2',label=permutedims(mylabel2))
plot!(xlabel="Time, days", ylabel = "Control")


In [ ]:
plot(tsamp./(60*40*24),r_soln1_noJ2'.-Re_mks,color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*40*24),r_soln2_noJ2'.-Re_mks,label=permutedims(mylabel2))
for i=1:num_sat
    println(r_soln1_noJ2[1]-Re_mks," ",params.Δa_max*a_mks*i)
    plot!([tsamp[1]./(60*40*24), tsamp[end]./(60*40*24)],[r_soln1_noJ2[1]-Re_mks-params.Δa_max*a_mks*i, r_soln1_noJ2[1]-Re_mks[1]-params.Δa_max*a_mks*i],color=:red, linewidth=2,label="Limit")
end
plot!(xlabel= "Time, days",ylabel= "Altitude, km")


In [ ]:
plot(tsamp[1:30:end]./(60*40*24),ϕ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),ϕ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Azimuth, rad")

In [ ]:
plot(tsamp[1:30:end]./(60*40*24),θ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),θ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Elevation, rad")

In [ ]:
# Final RAAN
broadcast.(rad2deg,ΔΩ_soln[:,end]')

println("Targeted Δθf ",  broadcast.(rad2deg, params.Δθf[1:num_sat]), " deg" )
println("Reached Δθf ",  broadcast.(rad2deg, Δθ_soln[:,end]), " deg" )


println("Targeted ΔΩ ",  broadcast.(rad2deg, params.ΔΩf[1:num_sat]), " deg" )
println("Reached ΔΩ ",  broadcast.(rad2deg, ΔΩ_soln[:,end]), " deg" )
println("Final Error ΔΩ ", broadcast.(rad2deg, (params.ΔΩf[1:num_sat] .- ΔΩ_soln[:,end])), " deg" )
println("Final Azimut Sat 1 ", broadcast.(rad2deg, ϕ_soln1[:,end]), " deg")
println("Final Azimut Sat Coupled ", broadcast.(rad2deg, ϕ_soln2[:,end]), " deg")

println("Final Elevation Sat 1 ", broadcast.(rad2deg, θ_soln1[:,end]), " deg")
println("Final Elevation Sat Coupled ", broadcast.(rad2deg, θ_soln2[:,end]), " deg")

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))


vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 



size_spacecraft = Vec(0.05,0.05,0.05)
size_plate = Vec(size_spacecraft[1]*3,0.01,0.04)

origin_spacecraft = Vec(x0_soln[1,1]/Re_mks-size_spacecraft[1]/2,x0_soln[2,1]/Re_mks-size_spacecraft[2]/2,x0_soln[3,1]/Re_mks-size_spacecraft[3]/2)
origin_plate = Vec(x0_soln[1,1]/Re_mks-size_plate[1]/2,x0_soln[2,1]/Re_mks-size_plate[2]/2,x0_soln[3,1]/Re_mks-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)

colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end

sleep(5)
for i = 1:length(x0_soln[1,:])-1
    for j = 1 : num_sat
        if j == 1
#             positions_1 = [x0_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x0_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x0_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
            positions_1 = [x0_soln[1+(6*j-6),i]/Re_mks-origin_spacecraft[1],x0_soln[2+(6*j-6),i]/Re_mks-origin_spacecraft[2],x0_soln[3+(6*j-6),i]/Re_mks-origin_spacecraft[3]]
            rotations_1 = i*2*pi/length(x0_soln[1+(6*j-6),:])
            position_1 = Translation(positions_1)
#             rotation_1 = LinearMap(RotZ(rotations_1))
            name_sat = "satellite"*string(j)
            settransform!(vis[name_sat], position_1)
#             settransform!(vis[name_sat], compose(position_1, rotation_1))
        end 
#         positions_2 = [x1_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x1_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x1_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
        positions_2 = [x1_soln[1+(6*j-6),i]/Re_mks-origin_spacecraft[1],x1_soln[2+(6*j-6),i]/Re_mks-origin_spacecraft[2],x1_soln[3+(6*j-6),i]/Re_mks-origin_spacecraft[3]]
#         rotations_2 = i*2*pi/length(x1_soln[1+(6*j-6),:])
        position_2 = Translation(positions_2)
#         rotation_2 = LinearMap(RotZ(rotations_2))
        name_sat = "satellite"*string(j+1)
        settransform!(vis[name_sat], position_2)
#         settransform!(vis[name_sat], compose(position_2, rotation_2))
    end
        #     settransform!(vis["plate1"], position)
    sleep(0.2)
end


In [ ]:
# using NaNStatistics
# window_size = length(tsamp)/1.0
# z1 = movmean(θ_soln1, window_size)
# z2 = movmean(θ_soln2, window_size)
# plot(tsamp,θ_soln1,linestyle=:dot)
# plot!(tsamp,θ_soln2,linestyle=:dot)
# plot!(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)

In [ ]:
# window_size = length(tsamp)/100.0
# z1 = movmean(Ω_soln1, window_size)
# z2 = movmean(Ω_soln2, window_size)
# # plot(tsamp,Ω_soln1,linestyle=:dot)
# # plot!(tsamp,Ω_soln2,linestyle=:dot)
# plot(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)
# plot!(ylim= [-1,1])

In [ ]:
k1/k2

In [ ]:
(k1/k2)*2*pi

In [ ]:
((7.0/2)*(Re*Re/(a*a))*J2*cosd(inc))*2*pi